# 📝 TiendaOnline - Bonus

- Se incorpora el loop `agregar_listado_productos` para subir de forma masiva un listado. No modifica precio, solo suma cantidades.
- Se incorpora `clientes` como atributo de la clase.
- Se incorpora la función `agregar_cliente`.
- Se incorpora la función `ver_clientes` y se importa la librería `tabulate`.
- Se incorpora la función `realizar_compra` compra interactiva.
- Se incorpora la función `procesar_pago` y `parse_moneda` para convertir el input previo a procesar el cambio.
- Se incorpora la función `_procesar_compra` para ejecutar internamete, `registrar_compra`y `parse_moneda` para convertir el input previo a procesar el cambio.
- Se incorpora la función `ver_compras_cliente` y `calcular_ventas_totales`.

In [ ]:
class TiendaOnLine:
    """ Clase que modela una tienda online con inventario, clientes y ventas.
    Permite gestionar productos, realizar compras y llevar registro de ventas. """
    
    def __init__(self, nombre):
         """ Constructor de la tienda. """
         self.nombre = nombre
         self.inventario = []                        # Lista de diccionarios: cada dict es {"nombre": str, "precio": float, "cantidad": int}
         self.ventas_totales = 0                     # Acumula el monto de todas las ventas
         self.clientes = {}                          # Dict {nombre_cliente: {"email": str, "compras": list}}

    
    def agregar_producto(self, nombre, precio, cantidad):
        # Validaciones
        if not nombre or not precio or not cantidad:
            print(f"Debes ingresar nombre, precio y cantidad del producto.\nEjecuta nuevamente.")
            return
        if precio <= 0:                                     #posibilidad de mejora, imposibilidad de ingreso de string.
            print("Debes ingresar un precio mayor a 0.")
            return
        if cantidad <= 0 or not isinstance(cantidad,int):   #posibilidad de mejora, imposibilidad de ingreso de string.
            print("Debes ingresar un número entero, mayor a 0.")
            return

        # Busqueda en el inventario
        for item in self.inventario:
            if item["nombre"] == nombre.title().strip():
                item["cantidad"] += cantidad
                print(f"Se actualizaron las unidades del producto '{nombre.title()}'. Hay {item['cantidad']:,.0f} unidad(es) disponible(s).")
                return
        
        # Si no existe, agregar nuevo producto
        nuevo_producto = {"nombre": nombre.title().strip(), "precio": precio, "cantidad": cantidad}
        self.inventario.append(nuevo_producto)
        print(f"El producto '{nombre.title().strip()}' fue agregado al inventario.")
        

           
    def agregar_listado_productos(self, listado_productos):
        for prod in listado_productos:
            self.agregar_producto(prod['nombre'], prod['precio'], prod['cantidad'])
    
    def ver_inventario(self):
        if not self.inventario:
            print("El inventario está vacío.")
        for producto in self.inventario:
            print(f"Nombre: {producto['nombre']}, Precio: ${producto['precio']:,.2f}, Cantidad: {producto['cantidad']:,.0f}")

    def buscar_producto(self, nombre):
        prod = nombre.title().strip()
        encontrado = False
        for producto in self.inventario:
            if prod == producto['nombre']:
                print(f"El producto '{prod}' se encuentra en el inventario.\nNombre: {producto['nombre']}, Precio: ${producto['precio']:,.2f}, Cantidad: {producto['cantidad']:,.0f}")
                encontrado = True
        if not encontrado:
            print(f"El producto '{prod}' no se encuentra en el inventario.")

    def actualizar_stock(self, nombre, cantidad):
        if not self.inventario:
            print(f"Debes ingresar un nombre de producto válido.")      #""""ERROR en el mensaje. corregir luego de examen: debe indicar que el inventario esta vacío"""
            return
        if not isinstance(cantidad, int):
            print(f"No puedes indicar cantidades con decimales, deben ser números enteros.")
            return
        if cantidad == 0:
            print("La cantidad ingresada es 0(cero). No se modificó el stock.") #"""ERROR falta return para cortar la funcion. corregir luego de examen: debe indicar que el inventario esta vacío"""
        for producto in self.inventario:
            if cantidad > 0:
                if nombre.title().strip() == producto['nombre']:
                    producto['cantidad'] += cantidad
                    print(f"Se actualizó el stock:\nNombre: {producto['nombre']}, Precio: ${producto['precio']:,.2f}, Cantidad: {producto['cantidad']:,.0f}")
                    break
            else:
                if nombre.title().strip() == producto['nombre'] and (-cantidad) > producto['cantidad']:
                    print(f"Sólo se pudo descontar {producto['cantidad']} unidad(es). El stock del producto está en 0(cero).")
                    producto['cantidad'] = 0
                    print(f"Nombre: {producto['nombre']}, Precio: ${producto['precio']:,.2f}, Cantidad: {producto['cantidad']:,.0f}")
                    break
        else:
            print(f"El producto '{nombre.title().strip()}', no se encuentra en el inventario.")

    def eliminar_producto(self, nombre):
        if not nombre:
            print("Debes ingresar un nombre de producto.")
        for producto in self.inventario:
            if nombre.title().strip() == producto['nombre']:
                self.inventario.remove(producto)
                print(f"El producto '{nombre}', se ha eliminado del inventario.")
                break
        else:
            print(f"El producto '{nombre.title().strip()}', no se encuentra en el inventario.")

    def calcular_valor_inventario(self):
        total_inventario = 0
        for producto in self.inventario:
            valor = producto['precio'] * producto['cantidad']
            total_inventario += valor
        print(f"El valor total del inventario es de ${total_inventario:,.2f}.\n")

### BONUS ###

    def agregar_cliente(self, nombre, correo):                              #POST EXAMEN INCLUIR POSIBILIDAD DE RESTAR STOCK, no estaba en la consigna.
        nom = nombre.title().strip()
        if not nombre or not correo:
            print("Debes ingresar un nombre de cliente y mail.")
            return    
        for cliente in self.clientes:
            if nom == cliente:
                print(f"El cliente {nom}, ya se encuentra registrado.")
                break 
        else:
            self.clientes[nom] = {"email": correo, "compras": []}
            print(self.clientes)
           
    def ver_clientes(self):
        from tabulate import tabulate
        data = [[nombre, datos['email'], len(datos['compras'])] for nombre, datos in self.clientes.items()]
        print("\n" + tabulate(data, headers=["Cliente", "Email", "Compras"], tablefmt="rounded_grid", colalign=("left","left","left")))

    # ---------- MÉTODO PRIVADO PARA PROCESAR COMPRA (EVITA DUPLICACIÓN) ----------
    def _procesar_compra(self, nombre_cliente, carrito):
        """
        Aplica los cambios de una compra ya confirmada.
        Retorna True si éxito, False si error.
        """
        nombre_cliente = nombre_cliente.title().strip()
        if nombre_cliente not in self.clientes:
            print(f"Cliente '{nombre_cliente}' no registrado.")
            return False
        if not carrito:
            print("Carrito vacío. No se procesa.")
            return False

        items_a_procesar = []
        total_compra = 0

        for producto, datos in carrito.items():
            nombre_prod = producto.title().strip()
            if isinstance(datos, dict):
                cantidad = datos.get('cantidad', 0)
                precio_carrito = datos.get('precio', None)
            else:
                cantidad = datos
                precio_carrito = None

            if cantidad <= 0:
                print(f"Cantidad inválida para '{nombre_prod}'. Se omite.")
                continue

            #buscar en inventario.
            prod_inv = None
            for p in self.inventario:
                if p['nombre'] == nombre_prod:
                    prod_inv = p
                    break
            if not prod_inv:
                print(f"Producto '{nombre_prod}' no existe. Compra cancelada.")
                return False
            if cantidad > prod_inv['cantidad']:
                print(f"Stock insuficiente para '{nombre_prod}'. Disponible: {prod_inv['cantidad']}. Compra cancelada.")
                return False

            precio = prod_inv['precio']
            if precio_carrito and precio_carrito > 0:
                precio = precio_carrito

            monto = precio * cantidad
            total_compra += monto
            items_a_procesar.append((prod_inv, cantidad, monto, nombre_prod))

        if not items_a_procesar:
            print("No hay productos válidos para procesar.")
            return False

        #aplica cambios
        for prod_inv, cant, monto, nombre_prod in items_a_procesar:
            prod_inv['cantidad'] -= cant
            self.ventas_totales += monto
            self.clientes[nombre_cliente]['compras'].append({
                'producto': nombre_prod,
                'cantidad': cant,
                'monto': monto
            })

        print(f"Compra procesada correctamente. Total: ${total_compra:,.2f}")
        return True

    # ---------- REALIZAR COMPRA (INTERACTIVO) ----------
    def realizar_compra(self, nombre_cliente):
        nombre_cliente = nombre_cliente.title().strip()
        if nombre_cliente not in self.clientes:
            print(f"El cliente '{nombre_cliente}' no está registrado.")
            return
        if not self.inventario:
            print("El inventario está vacío. No se pueden realizar compras.")
            return

        print("\nInventario Disponible:\n")
        for i, prod in enumerate(self.inventario, start=1):
            print(f"{i}. {prod['nombre']} - ${prod['precio']:.2f} (stock: {prod['cantidad']})")

        carrito = {}
        while True:
            print("\nCarrito:")
            if not carrito:
                print("(vacío)")
            else:
                total_parcial = 0
                for nombre, cantidad in carrito.items():
                    precio = next(p['precio'] for p in self.inventario if p['nombre'] == nombre)
                    subtotal = precio * cantidad
                    total_parcial += subtotal
                    print(f"{nombre} x{cantidad} = ${subtotal:,.2f}")
                print(f"\nTotal: ${total_parcial:,.2f}")

            comprar_producto = input("\n¿Qué producto quieres comprar? (número / nombre / 'salir'): ").strip()
            if comprar_producto.lower() == 'salir':
                break

            producto = None
            if comprar_producto.isdigit():
                indice = int(comprar_producto) - 1
                if 0 <= indice < len(self.inventario):
                    producto = self.inventario[indice]
            else:
                nombre_prod = comprar_producto.title().strip()
                for p in self.inventario:
                    if p['nombre'] == nombre_prod:
                        producto = p
                        break

            if not producto:
                print("Producto no encontrado.")
                continue

            try:
                cantidad = int(input(f"¿Cuántas unidades de '{producto['nombre']}' quieres comprar? "))
                if cantidad <= 0:
                    print("La cantidad debe ser un número positivo.")
                    continue
            except ValueError:
                print("Ingresa un número entero válido.")
                continue

            if cantidad > producto['cantidad']:
                print(f"Stock insuficiente. Solo hay {producto['cantidad']} unidades.")
                continue

            nombre_prod = producto['nombre']
            carrito[nombre_prod] = carrito.get(nombre_prod, 0) + cantidad
            print(f"{cantidad} x '{nombre_prod}' agregado al carrito.")

        if not carrito:
            print("No se agregaron productos. Compra cancelada.")
            return

        print("\nResumen de Compra:")
        total_compra = 0
        for nombre, cant in carrito.items():
            precio = next(p['precio'] for p in self.inventario if p['nombre'] == nombre)
            subtotal = precio * cant
            total_compra += subtotal
            print(f"{nombre} x{cant} = ${subtotal:,.2f}")
        print(f"Total a Pagar: ${total_compra:,.2f}")

        confirmar = input("\n¿Confirmar compra? (s/n): ").strip().lower()
        if confirmar != 's':
            print("Compra cancelada.")
            return

        if self._procesar_compra(nombre_cliente, carrito):
            print(f"\n¡Tu compra se realizó exitosamente!")
            print(f"Ventas totales acumuladas: ${self.ventas_totales:,.2f}")

    
    # ---------- REGISTRAR COMPRA (DESDE UN CARRITO EXTERNO) ----------
    def registrar_compra(self, nombre_cliente, carrito):
        """Registra una compra directamente (sin interacción). El carrito puede ser {'producto': cantidad} o {'producto': {'precio':p, 'cantidad':c}}."""
        if self._procesar_compra(nombre_cliente, carrito):
            print("Compra registrada exitosamente.")
        else:
            print("No se pudo registrar la compra.")

    @staticmethod
    def parse_moneda(cadena):
        cadena = cadena.strip().replace(' ', '')
        if not cadena:
            raise ValueError("Cadena vacía")
        if ',' in cadena and '.' in cadena:
            ultimo_comma = cadena.rfind(',')
            ultimo_punto = cadena.rfind('.')
            if ultimo_comma > ultimo_punto:
                cadena = cadena.replace('.', '')
                cadena = cadena.replace(',', '.')
            else:
                cadena = cadena.replace(',', '')
        elif ',' in cadena:
            cadena = cadena.replace(',', '.')
        return float(cadena)

    def procesar_pago(self):
        try:
            monto_str = input("Ingrese el monto total de la compra: ")
            pago_str = input("Ingrese el monto del pago: ")

            monto_total_compra = self.parse_moneda(monto_str)
            pago = self.parse_moneda(pago_str)

            if monto_total_compra <= 0 or pago <= 0:
                print("Debes ingresar montos mayores a 0.")
                return

            if monto_total_compra > pago:
                print(f"El monto de pago ${pago:,.2f} es inferior al de la compra ${monto_total_compra:,.2f}.")
                return
            else:
                cambio = pago - monto_total_compra
                print(f"El cambio es de ${cambio:,.2f}.")
                return

        except ValueError:
            print("Debes ingresar un número válido.")

    def ver_compras_cliente(self, nombre_cliente):

        nombre_cliente = nombre_cliente.title().strip()
        
        if nombre_cliente not in self.clientes:
            print(f"El cliente '{nombre_cliente}' no está registrado.")
            return
        compras = self.clientes[nombre_cliente]['compras']
        if not compras:
            print(f"{nombre_cliente} no ha realizado ninguna compra aún.")
            return
        print(f"\nHistorial de compras de {nombre_cliente}:\n")
        total_gastado = 0
        for idx, compra in enumerate(compras, 1):
            print(f"  {idx}. Producto: {compra['producto']}, Cantidad: {compra['cantidad']}, Monto: ${compra['monto']:,.2f}")
            total_gastado += compra['monto']
        print()
        print(f"Total gastado: ${total_gastado:,.2f}")

    def calcular_ventas_totales(self):
        print(f"Ventas totales de la tienda: ${self.ventas_totales:,.2f}")

IndentationError: unindent does not match any outer indentation level (<string>, line 8)

In [3]:
farma=TiendaOnLine("Farma_Salud")

In [4]:
farma.agregar_listado_productos([{'nombre': 'Vitamina C 1000mg (30 tabs)', 'precio': 12.50, 'cantidad': 150},
    {'nombre': 'Gel antibacterial (500ml)', 'precio': 8.99, 'cantidad': 300},
    {'nombre': 'Termómetro digital', 'precio': 15.90, 'cantidad': 80},
    {'nombre': 'Mascarillas KN95 (10 uds)', 'precio': 9.99, 'cantidad': 500},
    {'nombre': 'Jabón de avena hipoalergénico', 'precio': 4.50, 'cantidad': 200},
    {'nombre': 'Colágeno hidrolizado (300g)', 'precio': 29.99, 'cantidad': 60},
    {'nombre': 'Vendas elásticas (2 uds)', 'precio': 7.20, 'cantidad': 120},
    {'nombre': 'Probióticos 50 mil millones', 'precio': 34.90, 'cantidad': 45},
    {'nombre': 'Nebulizador portátil', 'precio': 49.99, 'cantidad': 25},
    {'nombre': 'Pillbox organizador semanal', 'precio': 6.75, 'cantidad': 180}]
)

El producto 'Vitamina C 1000Mg (30 Tabs)' fue agregado al inventario.
El producto 'Gel Antibacterial (500Ml)' fue agregado al inventario.
El producto 'Termómetro Digital' fue agregado al inventario.
El producto 'Mascarillas Kn95 (10 Uds)' fue agregado al inventario.
El producto 'Jabón De Avena Hipoalergénico' fue agregado al inventario.
El producto 'Colágeno Hidrolizado (300G)' fue agregado al inventario.
El producto 'Vendas Elásticas (2 Uds)' fue agregado al inventario.
El producto 'Probióticos 50 Mil Millones' fue agregado al inventario.
El producto 'Nebulizador Portátil' fue agregado al inventario.
El producto 'Pillbox Organizador Semanal' fue agregado al inventario.


In [175]:
farma.ver_inventario()

Nombre: Vitamina C 1000Mg (30 Tabs), Precio: $12.50, Cantidad: 150
Nombre: Gel Antibacterial (500Ml), Precio: $8.99, Cantidad: 300
Nombre: Termómetro Digital, Precio: $15.90, Cantidad: 80
Nombre: Mascarillas Kn95 (10 Uds), Precio: $9.99, Cantidad: 500
Nombre: Jabón De Avena Hipoalergénico, Precio: $4.50, Cantidad: 200
Nombre: Colágeno Hidrolizado (300G), Precio: $29.99, Cantidad: 60
Nombre: Vendas Elásticas (2 Uds), Precio: $7.20, Cantidad: 120
Nombre: Probióticos 50 Mil Millones, Precio: $34.90, Cantidad: 45
Nombre: Nebulizador Portátil, Precio: $49.99, Cantidad: 25
Nombre: Pillbox Organizador Semanal, Precio: $6.75, Cantidad: 180


In [176]:
print(type(farma.clientes))
print(len(farma.clientes))
print(farma.clientes)

<class 'dict'>
0
{}


In [177]:
farma.agregar_cliente("","prueba@email.com")

Debes ingresar un nombre de cliente y mail.


In [178]:
farma.agregar_cliente("Luisa Trujillo","bea@gmail.com")

{'Luisa Trujillo': {'email': 'bea@gmail.com', 'compras': []}}


In [179]:
print(farma.clientes)

{'Luisa Trujillo': {'email': 'bea@gmail.com', 'compras': []}}


In [180]:
farma.ver_clientes()


╭────────────────┬───────────────┬───────────╮
│ Cliente        │ Email         │ Compras   │
├────────────────┼───────────────┼───────────┤
│ Luisa Trujillo │ bea@gmail.com │ 0         │
╰────────────────┴───────────────┴───────────╯


In [181]:
farma.agregar_cliente("Crono Bunge", "crono@hotmail.com")

{'Luisa Trujillo': {'email': 'bea@gmail.com', 'compras': []}, 'Crono Bunge': {'email': 'crono@hotmail.com', 'compras': []}}


In [182]:
farma.realizar_compra("Crono Bunge")


Inventario Disponible:

1. Vitamina C 1000Mg (30 Tabs) - $12.50 (stock: 150)
2. Gel Antibacterial (500Ml) - $8.99 (stock: 300)
3. Termómetro Digital - $15.90 (stock: 80)
4. Mascarillas Kn95 (10 Uds) - $9.99 (stock: 500)
5. Jabón De Avena Hipoalergénico - $4.50 (stock: 200)
6. Colágeno Hidrolizado (300G) - $29.99 (stock: 60)
7. Vendas Elásticas (2 Uds) - $7.20 (stock: 120)
8. Probióticos 50 Mil Millones - $34.90 (stock: 45)
9. Nebulizador Portátil - $49.99 (stock: 25)
10. Pillbox Organizador Semanal - $6.75 (stock: 180)

Carrito:
(vacío)
15 x 'Mascarillas Kn95 (10 Uds)' agregado al carrito.

Carrito:
Mascarillas Kn95 (10 Uds) x15 = $149.85

Total: $149.85
Producto no encontrado.

Carrito:
Mascarillas Kn95 (10 Uds) x15 = $149.85

Total: $149.85
3 x 'Jabón De Avena Hipoalergénico' agregado al carrito.

Carrito:
Mascarillas Kn95 (10 Uds) x15 = $149.85
Jabón De Avena Hipoalergénico x3 = $13.50

Total: $163.35
Producto no encontrado.

Carrito:
Mascarillas Kn95 (10 Uds) x15 = $149.85
Jabón D

In [183]:
farma.realizar_compra("Luisa Trujillo")


Inventario Disponible:

1. Vitamina C 1000Mg (30 Tabs) - $12.50 (stock: 150)
2. Gel Antibacterial (500Ml) - $8.99 (stock: 300)
3. Termómetro Digital - $15.90 (stock: 80)
4. Mascarillas Kn95 (10 Uds) - $9.99 (stock: 485)
5. Jabón De Avena Hipoalergénico - $4.50 (stock: 197)
6. Colágeno Hidrolizado (300G) - $29.99 (stock: 60)
7. Vendas Elásticas (2 Uds) - $7.20 (stock: 120)
8. Probióticos 50 Mil Millones - $34.90 (stock: 35)
9. Nebulizador Portátil - $49.99 (stock: 25)
10. Pillbox Organizador Semanal - $6.75 (stock: 180)

Carrito:
(vacío)
5 x 'Vendas Elásticas (2 Uds)' agregado al carrito.

Carrito:
Vendas Elásticas (2 Uds) x5 = $36.00

Total: $36.00

Resumen de Compra:
Vendas Elásticas (2 Uds) x5 = $36.00
Total a Pagar: $36.00
Compra cancelada.


In [184]:
farma.ver_inventario()

Nombre: Vitamina C 1000Mg (30 Tabs), Precio: $12.50, Cantidad: 150
Nombre: Gel Antibacterial (500Ml), Precio: $8.99, Cantidad: 300
Nombre: Termómetro Digital, Precio: $15.90, Cantidad: 80
Nombre: Mascarillas Kn95 (10 Uds), Precio: $9.99, Cantidad: 485
Nombre: Jabón De Avena Hipoalergénico, Precio: $4.50, Cantidad: 197
Nombre: Colágeno Hidrolizado (300G), Precio: $29.99, Cantidad: 60
Nombre: Vendas Elásticas (2 Uds), Precio: $7.20, Cantidad: 120
Nombre: Probióticos 50 Mil Millones, Precio: $34.90, Cantidad: 35
Nombre: Nebulizador Portátil, Precio: $49.99, Cantidad: 25
Nombre: Pillbox Organizador Semanal, Precio: $6.75, Cantidad: 180


In [185]:
farma.realizar_compra("Luisa Trujillo")


Inventario Disponible:

1. Vitamina C 1000Mg (30 Tabs) - $12.50 (stock: 150)
2. Gel Antibacterial (500Ml) - $8.99 (stock: 300)
3. Termómetro Digital - $15.90 (stock: 80)
4. Mascarillas Kn95 (10 Uds) - $9.99 (stock: 485)
5. Jabón De Avena Hipoalergénico - $4.50 (stock: 197)
6. Colágeno Hidrolizado (300G) - $29.99 (stock: 60)
7. Vendas Elásticas (2 Uds) - $7.20 (stock: 120)
8. Probióticos 50 Mil Millones - $34.90 (stock: 35)
9. Nebulizador Portátil - $49.99 (stock: 25)
10. Pillbox Organizador Semanal - $6.75 (stock: 180)

Carrito:
(vacío)
Producto no encontrado.

Carrito:
(vacío)
4 x 'Jabón De Avena Hipoalergénico' agregado al carrito.

Carrito:
Jabón De Avena Hipoalergénico x4 = $18.00

Total: $18.00
63 x 'Vendas Elásticas (2 Uds)' agregado al carrito.

Carrito:
Jabón De Avena Hipoalergénico x4 = $18.00
Vendas Elásticas (2 Uds) x63 = $453.60

Total: $471.60
10 x 'Gel Antibacterial (500Ml)' agregado al carrito.

Carrito:
Jabón De Avena Hipoalergénico x4 = $18.00
Vendas Elásticas (2 Uds)

In [186]:
farma.ver_clientes()


╭────────────────┬───────────────────┬───────────╮
│ Cliente        │ Email             │ Compras   │
├────────────────┼───────────────────┼───────────┤
│ Luisa Trujillo │ bea@gmail.com     │ 4         │
├────────────────┼───────────────────┼───────────┤
│ Crono Bunge    │ crono@hotmail.com │ 3         │
╰────────────────┴───────────────────┴───────────╯


In [187]:
farma.procesar_pago()


El monto de pago $50.00 es inferior al de la compra $100.00.


In [188]:
farma.procesar_pago()

Debes ingresar montos mayores a 0.


In [189]:
farma.procesar_pago()

Debes ingresar un número válido.


In [190]:
farma.procesar_pago()

El cambio es de $50.00.


In [191]:
carrito = {"carrito":{"Vitamina C 1000 Mg (30 Tabs)": 1, "Vendas Elásticas (2 Uds)": 2}}
farma.registrar_compra("Juana", carrito)

Cliente 'Juana' no registrado.
No se pudo registrar la compra.


In [192]:
carrito = {"Vitamina C 1000Mg (30 Tabs)":{"precio":12.50,"cantidad": 1}, "Vendas Elásticas (2 Uds)":{"precio":7.20,"cantidad":2}}
farma.registrar_compra("Crono Bunge", carrito)

Compra procesada correctamente. Total: $26.90
Compra registrada exitosamente.


In [193]:
carrito = {"Probióticos 50 Mil Millones" : 3}
farma.registrar_compra("Luisa Trujillo", carrito)

Compra procesada correctamente. Total: $104.70
Compra registrada exitosamente.


In [194]:
carrito = {"Probióticos Mil Millones" : 3}
farma.registrar_compra("Luisa Trujillo", carrito)

Producto 'Probióticos Mil Millones' no existe. Compra cancelada.
No se pudo registrar la compra.


In [195]:
farma.ver_compras_cliente("Luisa Trujillo")


Historial de compras de Luisa Trujillo:

  1. Producto: Jabón De Avena Hipoalergénico, Cantidad: 4, Monto: $18.00
  2. Producto: Vendas Elásticas (2 Uds), Cantidad: 63, Monto: $453.60
  3. Producto: Gel Antibacterial (500Ml), Cantidad: 10, Monto: $89.90
  4. Producto: Vitamina C 1000Mg (30 Tabs), Cantidad: 3, Monto: $37.50
  5. Producto: Probióticos 50 Mil Millones, Cantidad: 3, Monto: $104.70

Total gastado: $703.70


In [196]:
farma.ver_compras_cliente("12123124")

El cliente '12123124' no está registrado.


In [197]:
farma.ver_compras_cliente("CroNo BUnge")


Historial de compras de Crono Bunge:

  1. Producto: Mascarillas Kn95 (10 Uds), Cantidad: 15, Monto: $149.85
  2. Producto: Jabón De Avena Hipoalergénico, Cantidad: 3, Monto: $13.50
  3. Producto: Probióticos 50 Mil Millones, Cantidad: 10, Monto: $349.00
  4. Producto: Vitamina C 1000Mg (30 Tabs), Cantidad: 1, Monto: $12.50
  5. Producto: Vendas Elásticas (2 Uds), Cantidad: 2, Monto: $14.40

Total gastado: $539.25


In [6]:
farma.calcular_ventas_totales()

Ventas totales de la tienda: $0.00


In [9]:
farma.actualizar_stock("Mascarillas Kn95 (10 Uds)",10)

Se actualizó el stock:
Nombre: Mascarillas Kn95 (10 Uds), Precio: $9.99, Cantidad: 520
